In [ ]:
# Haystack 2.x + the only extras this notebook actually needs.
# LLMDocumentContentExtractor renders PDF/image pages, which needs pypdfium2 + pillow.
# (Removed unused/conflicting packages: chromadb, chroma-haystack, langchain*, PyPDF2,
#  and the pinned pydantic/chromadb<0.5 lines that tended to break the environment.)
!pip install -q haystack-ai pypdfium2 pillow


  Using cached chromadb-0.4.24-py3-none-any.whl.metadata (7.3 kB)
Using cached chromadb-0.4.24-py3-none-any.whl (525 kB)
  Attempting uninstall: chromadb
    Found existing installation: chromadb 1.5.9
    Uninstalling chromadb-1.5.9:
      Successfully uninstalled chromadb-1.5.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chroma-haystack 4.3.0 requires chromadb>=1.5.4, but you have chromadb 0.4.24 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 3.6 MB/s eta 0:00:00


## 📂 Important Step: Upload the sample files

Before running the code cells, upload the two sample files this notebook uses:

👉 `file-sample_150kB.pdf` — a small PDF with text, a table and a figure
👉 `images.png` — an image containing printed text (for OCR-style extraction)

---

### 🔼 How to upload in Google Colab
1. In the **left sidebar**, click the 📁 **Files** tab.
2. Click the **Upload** button.
3. Select `file-sample_150kB.pdf` and `images.png` from your computer.
4. They should appear under `/content/`.

---

### ⚠️ Notes
- Keep the exact filenames — the code expects `/content/file-sample_150kB.pdf` and `/content/images.png`.
- If you restart the Colab runtime, re-upload the files.


### 🔑 Environment Configuration

Before initializing the model, make sure to **set the following environment variables** with your correct Azure values:  

```python
os.environ['AZURE_OPENAI_API_KEY'] = "<your-api-key>"
os.environ['AZURE_OPENAI_ENDPOINT'] = "<your-endpoint-url>"
os.environ['AZURE_OPENAI_API_VERSION'] = "<supported-api-version>"
os.environ['AZURE_OPENAI_LLM_DEPLOYMENT'] = "<your-deployment-name>"


In [ ]:
import os

# ── Azure OpenAI configuration ───────────────────────────────────
# Fill these in with YOUR Azure OpenAI values. Never commit a real key to a notebook.
os.environ["AZURE_OPENAI_API_KEY"] = "<your-azure-openai-api-key>"

# IMPORTANT: azure_endpoint must be the BASE resource URL only, ending in a slash —
# NOT the full ".../openai/deployments/<name>/chat/completions?api-version=..." path.
# Correct shape:  https://<your-resource>.openai.azure.com/
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<your-resource>.openai.azure.com/"

# An API version your Azure resource supports.
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-12-01-preview"

# Your Azure *deployment name*. It must be a VISION-capable model (e.g. gpt-4o,
# gpt-4o-mini, gpt-4.1-mini), since the content extractor sends page images.
os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"] = "<your-deployment-name>"

# Convenience aliases used by a later cell
GPT4O_DEPLOYMENT_NAME    = os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"]
GPT4O_AZURE_ENDPOINT     = os.environ["AZURE_OPENAI_ENDPOINT"]
GPT4O_OPENAI_API_KEY     = os.environ["AZURE_OPENAI_API_KEY"]
GPT4O_OPENAI_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]

# Note: the previous version of this cell also created a LangChain AzureChatOpenAI
# client (with an invalid model name and the wrong endpoint shape). It was unused —
# every step below uses Haystack's AzureOpenAIChatGenerator — so it has been removed.


In [ ]:
!pip install pypdfium2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 36.9 MB/s eta 0:00:00


# If the error appears again after installing **pypdfium2**, restart the session and rerun the GPT environment configuration.

## LLMDocumentContentExtractor

In [ ]:
from haystack import Document
import os
from haystack.utils import Secret
from haystack.components.generators.chat import AzureOpenAIChatGenerator
from haystack.components.extractors.image import LLMDocumentContentExtractor

from haystack.utils import Secret
from haystack.components.generators.chat import AzureOpenAIChatGenerator

chat_generator = AzureOpenAIChatGenerator(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
    azure_deployment=GPT4O_DEPLOYMENT_NAME,
    api_version=GPT4O_OPENAI_API_VERSION
)

# Create the extractor
extractor = LLMDocumentContentExtractor(
    chat_generator=chat_generator,
    file_path_meta_field="file_path",
    raise_on_failure=False
)

# Create documents with image file paths
documents = [
    # Document(content="", meta={"file_path": "/content/2408.09869v1.pdf"}),
    Document(content="", meta={"file_path": "/content/file-sample_150kB.pdf", "page_number": 1})
]

# Run the extractor
result = extractor.run(documents=documents)

# Check results
print(f"Successfully processed: {len(result['documents'])}")
print(f"Failed documents: {len(result['failed_documents'])}")

# Access extracted content
for doc in result["documents"]:
    print(f"File: {doc.meta['file_path']}")
    print("Extracted content:\n")
    print(doc.content)  # full text
    print("\n" + "="*80 + "\n")  # optional separator between documents


Successfully processed: 1
Failed documents: 0
File: /content/file-sample_150kB.pdf
Extracted content:

# Lorem ipsum

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Nunc ac faucibus odio.

Vestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut varius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum condimentum. **Vivamus dapibus sodales ex, vitae malesuada ipsum cursus convallis. Maecenas sed egestas nulla, ac condimentum orci.** Mauris diam felis, vulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus nisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum, ac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit amet tortor quis risus auctor condimentum. Morbi in ullamcorper elit. Nulla iaculis tellus sit amet mauris tempus fringilla.

Maecenas mauris lectus, lobortis et purus mattis, blandit dictum t

In [ ]:
from haystack import Document
from haystack.components.extractors.image import LLMDocumentContentExtractor
from haystack.components.generators.chat import AzureOpenAIChatGenerator
from haystack.utils import Secret
import os

# Custom prompt
custom_prompt = """
Extract all text content from this image-based document.

Instructions:
- Extract text exactly as it appears
- Preserve the reading order
- Format tables as markdown
- Describe any images or diagrams briefly
- Maintain document structure

Document:
"""

# Initialize Azure OpenAI chat generator
chat_generator = AzureOpenAIChatGenerator(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),  # reads key from env
    azure_deployment=os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"],  # e.g., gpt-5-mini
    api_version=os.environ["AZURE_OPENAI_API_VERSION"]  # e.g., "2025-04-01-preview"
)

# Create the extractor
extractor = LLMDocumentContentExtractor(
    chat_generator=chat_generator,
    prompt=custom_prompt,
    file_path_meta_field="file_path",
    raise_on_failure=False
)

# Create documents (PDF example with page_number)
documents = [
    Document(content="", meta={"file_path": "/content/images.png"}),

    # add more pages if needed
]

# Run the extractor
result = extractor.run(documents=documents)

# Print results
for doc in result["documents"]:
    print(f"File: {doc.meta['file_path']} (Page {doc.meta.get('page_number', 'N/A')})")
    print("Extracted content:\n")
    print(doc.content)
    print("\n" + "="*80 + "\n")


File: /content/images.png (Page N/A)
Extracted content:

# HOW TO COMBINE  
## TEXT AND IMAGE  
### IN ELEARNING DESIGN




In [ ]:
from haystack import Pipeline
from haystack.components.extractors.image import LLMDocumentContentExtractor
from haystack.components.generators.chat import AzureOpenAIChatGenerator
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.writers import DocumentWriter
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document
from haystack.utils import Secret
import os

# Create document store
document_store = InMemoryDocumentStore()

# Initialize Azure OpenAI chat generator
azure_chat_generator = AzureOpenAIChatGenerator(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
    azure_deployment=os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"]
)

# Create pipeline
p = Pipeline()
p.add_component(
    instance=LLMDocumentContentExtractor(
        chat_generator=azure_chat_generator,
        file_path_meta_field="file_path"
    ),
    name="content_extractor"
)
p.add_component(instance=DocumentSplitter(), name="splitter")
p.add_component(instance=DocumentWriter(document_store=document_store), name="writer")

# Connect components
p.connect("content_extractor.documents", "splitter.documents")
p.connect("splitter.documents", "writer.documents")

# Create test documents
docs = [
    Document(content="", meta={"file_path": "/content/file-sample_150kB.pdf", "page_number": 1}),
    Document(content="", meta={"file_path": "/content/images.png"})
]

# Run pipeline
result = p.run({"content_extractor": {"documents": docs}})

# Check results
# Safe access for processed documents
processed_docs = result.get("content_extractor", {}).get("documents", [])
failed_docs = result.get("content_extractor", {}).get("failed_documents", [])


print(f"Successfully processed: {len(processed_docs)}")
print(f"Failed documents: {len(failed_docs)}")


# Access documents in the store
stored_docs = document_store.filter_documents()
print(f"Documents in store: {len(stored_docs)}")

# for doc in stored_docs:
#     print(f"File: {doc.meta.get('file_path')}")
#     print("Extracted content:\n")
#     print(doc.content)
#     print("\n" + "="*80 + "\n")



Successfully processed: 0
Failed documents: 0
Documents in store: 3


In [ ]:
for doc in stored_docs:
    print(f"File: {doc.meta.get('file_path')}")
    print("Extracted content:\n")
    print(doc.content)
    print("\n" + "="*80 + "\n")

File: /content/file-sample_150kB.pdf
Extracted content:

Lorem ipsum

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Nunc ac faucibus odio.

Vestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut varius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum condimentum. Vivamus dapibus sodales ex, vitae malesuada ipsum cursus convallis. Maecenas sed egestas nulla, ac condimentum orci. Mauris diam felis, vulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus nisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum, ac accumsan nunc vehicula vitae. Nulla ege justo in felis tristique fringilla. Morbi sit amet tortor quis risus auctor condimentum. Morbi in ullamcorper elit. Nulla iaculis tellus sit amet mauris tempus fringing.

Maecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus.

- Maecenas non lorem quis tellus placerat vari

## LLMMetadataExtractor

In [ ]:
from haystack import Document
from haystack.components.extractors.llm_metadata_extractor import LLMMetadataExtractor
from haystack.components.generators.chat import AzureOpenAIChatGenerator
from haystack.utils import Secret
import os

# Example documents
docs = [
    Document(content="deepset was founded in 2018 in Berlin, and is known for its Haystack framework"),
    Document(content="Hugging Face is a company founded in New York, USA and is known for its Transformers library")
]

# Initialize Azure OpenAI Chat Generator
azure_chat_generator = AzureOpenAIChatGenerator(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
    azure_deployment=os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"]
)

# Make sure you have your custom prompt for NER
NER_PROMPT = '''
    -Goal-
    Given text and a list of entity types, identify all entities of those types from the text.

    -Steps-
    1. Identify all entities. For each identified entity, extract the following information:
    - entity_name: Name of the entity, capitalized
    - entity_type: One of the following types: [organization, product, service, industry]
    Format each entity as a JSON like: {"entity": <entity_name>, "entity_type": <entity_type>}

    2. Return output in a single list with all the entities identified in steps 1.

    -Examples-
    ######################
    Example 1:
    entity_types: [organization, person, partnership, financial metric, product, service, industry, investment strategy, market trend]
    text: Another area of strength is our co-brand issuance. Visa is the primary network partner for eight of the top
    10 co-brand partnerships in the US today and we are pleased that Visa has finalized a multi-year extension of
    our successful credit co-branded partnership with Alaska Airlines, a portfolio that benefits from a loyal customer
    base and high cross-border usage.
    We have also had significant co-brand momentum in CEMEA. First, we launched a new co-brand card in partnership
    with Qatar Airways, British Airways and the National Bank of Kuwait. Second, we expanded our strong global
    Marriott relationship to launch Qatar's first hospitality co-branded card with Qatar Islamic Bank. Across the
    United Arab Emirates, we now have exclusive agreements with all the leading airlines marked by a recent
    agreement with Emirates Skywards.
    And we also signed an inaugural Airline co-brand agreement in Morocco with Royal Air Maroc. Now newer digital
    issuers are equally
    ------------------------
    output:
    {"entities": [{"entity": "Visa", "entity_type": "company"}, {"entity": "Alaska Airlines", "entity_type": "company"}, {"entity": "Qatar Airways", "entity_type": "company"}, {"entity": "British Airways", "entity_type": "company"}, {"entity": "National Bank of Kuwait", "entity_type": "company"}, {"entity": "Marriott", "entity_type": "company"}, {"entity": "Qatar Islamic Bank", "entity_type": "company"}, {"entity": "Emirates Skywards", "entity_type": "company"}, {"entity": "Royal Air Maroc", "entity_type": "company"}]}
    #############################
    -Real Data-
    ######################
    entity_types: [company, organization, person, country, product, service]
    text: {{ document.content }}
    ######################
    output:
    '''

# Initialize LLM Metadata Extractor
extractor = LLMMetadataExtractor(
    prompt=NER_PROMPT,
    chat_generator=azure_chat_generator,
    expected_keys=["entities"],
    raise_on_failure=False,
)

# Optional: warm up the LLM
extractor.warm_up()

# Run extractor
result = extractor.run(documents=docs)

# Inspect results
for doc in result["documents"]:
    print(f"Text: {doc.content}")
    print(f"Extracted metadata: {doc.meta}")
    print("\n" + "="*60 + "\n")


  {
    "entity": "Deepset",
    "entity_type": "company"
  },
  {
    "entity": "Berlin",
    "entity_type": "country"
  },
  {
    "entity": "Haystack",
    "entity_type": "product"
  }
]
  {"entity": "Hugging Face", "entity_type": "company"},
  {"entity": "New York", "entity_type": "city"},
  {"entity": "USA", "entity_type": "country"}
]
